# Vgg16 Model

# Environment

In [ ]:
# This mounts your Google Drive to the Colab VM.
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cse493g1/assignments/assignment5/'
FOLDERNAME = 'cse493g/project/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Create a local folder on the Colab VM
!mkdir -p /content/local_data

# Copy the HDF5 files from Google Drive to the fast local disk
# to save training time as currently drive file access is the bottleneck
!cp -r /content/drive/MyDrive/project/data/*.h5 /content/local_data/

print("Copy complete!")

Copy complete!


In [2]:
import os
import random
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.metrics import roc_auc_score

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Device ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


# Config

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR = "/content/local_data"        # directory containing the .h5 files
CHECKPOINT_PATH = "/content/best_vgg16.pt"

# ── File names ─────────────────────────────────────────────────────────────────
FILES = {
    "train": {
        "x": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_train_x.h5"),
        "y": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_train_y.h5"),
    },
    "val": {
        "x": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_valid_x.h5"),
        "y": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_valid_y.h5"),
    },
    "test": {
        "x": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_x.h5"),
        "y": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_y.h5"),
    },
}
# ── File names ─────────────────────────────────────────────────────────────────
FILES = {
    "train": {
        "x": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_train_x.h5"),
        "y": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_train_y.h5"),
    },
    "val": {
        "x": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_valid_x.h5"),
        "y": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_valid_y.h5"),
    },
    "test": {
        "x": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_x.h5"),
        "y": os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_y.h5"),
    },
}

# Hyperparameters
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.5
BATCH_SIZE    = 64

# Architecture
BACKBONE_OUT_DIM = 4096  # VGG16 last FC output
HEAD_HIDDEN_DIM  = 128

# Checkpoint path
CHECKPOINT_PATH = "./best_vgg16.pt"

print("Configuration loaded.")
print(f"Epochs      : {NUM_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Dropout rate : {DROPOUT_RATE}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Batch   : {BATCH_SIZE}")

Configuration loaded.
Epochs      : 10
Learning rate: 0.001
Dropout rate : 0.5
Checkpoint   : ./best_vgg16.pt
Batch   : 64


# Dataset

In [ ]:
class PCamDataset(Dataset):
    """Lazy-loading Dataset for PatchCamelyon HDF5 files."""

    def __init__(self, x_path: str, y_path: str, transform=None):
        self.x_path = x_path
        self.y_path = y_path
        self.transform = transform
        # Read length only — do not load data into memory
        with h5py.File(x_path, "r") as f:
            self.length = f["x"].shape[0]

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.x_path, "r") as fx:
            image = fx["x"][idx]                          # (96, 96, 3) uint8
        with h5py.File(self.y_path, "r") as fy:
            label = int(fy["y"][idx].squeeze())           # scalar int

        # Normalise to [0, 1] and convert to (C, H, W) float tensor
        image = image.astype(np.float32) / 255.0
        image = torch.tensor(image).permute(2, 0, 1)     # (3, 96, 96)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


# ── Transforms ─────────────────────────────────────────────────────────────────
_imagenet_norm = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    _imagenet_norm,
])

eval_transform = transforms.Compose([
    _imagenet_norm,
])

# ── Instantiate datasets ────────────────────────────────────────────────────────
train_dataset = PCamDataset(FILES["train"]["x"], FILES["train"]["y"], transform=train_transform)
val_dataset   = PCamDataset(FILES["val"]["x"],   FILES["val"]["y"],   transform=eval_transform)
test_dataset  = PCamDataset(FILES["test"]["x"],  FILES["test"]["y"],  transform=eval_transform)

print(f"Dataset sizes  →  train: {len(train_dataset):,}  |  val: {len(val_dataset):,}  |  test: {len(test_dataset):,}")

# ── DataLoaders ────────────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Batches per epoch  →  train: {len(train_loader):,}  |  val: {len(val_loader):,}  |  test: {len(test_loader):,}")


Dataset sizes  →  train: 262,144  |  val: 32,768  |  test: 32,768
Batches per epoch  →  train: 4,096  |  val: 512  |  test: 512


# Model

In [ ]:
# --- Backbone ---
backbone = models.vgg16(pretrained=True)

# Freeze backbone parameters
for param in backbone.parameters():
    param.requires_grad = False

# Remove last FC from classifier
backbone.classifier = nn.Sequential(*list(backbone.classifier.children())[:-1])  # output (batch, 4096)

# --- Classification Head ---
head = nn.Sequential(
    nn.Linear(BACKBONE_OUT_DIM, HEAD_HIDDEN_DIM),
    nn.ReLU(),
    nn.Dropout(DROPOUT_RATE),
    nn.Linear(HEAD_HIDDEN_DIM, 1),
    nn.Sigmoid()
)

# --- Full Model ---
class VGG16Classifier(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head
    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

model = VGG16Classifier(backbone, head).to(DEVICE)

# --- Parameter Check ---
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
frozen    = total - trainable
print(f"Total params    : {total:,}")
print(f"Frozen (backbone): {frozen:,}")
print(f"Trainable (head): {trainable:,}")
assert trainable < total, "Backbone parameters should be frozen."

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 242MB/s]


Total params    : 134,785,089
Frozen (backbone): 134,260,544
Trainable (head): 524,545


# Training/Vaidation

In [ ]:
# ── Loss & Optimizer ───────────────────────────────────────────────────────────
# BCELoss is used with Sigmoid output in the head
criterion = nn.BCELoss()

# Pass ONLY trainable (head) parameters to the optimizer
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = (outputs >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        batch_acc = (preds == labels).float().mean().item()

        if batch_idx % 100 == 0:
            batch_acc = (preds == labels).float().mean().item()
            print(
                f"Batch {batch_idx}/{len(loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Batch Acc: {batch_acc:.4f}"
            )


    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def train(model, train_loader, val_loader, criterion, optimizer,
          num_epochs, device, checkpoint_path):
    history = {"train_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate(model, val_loader, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        saved = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)
            saved = "  ← best checkpoint saved"

        print(
            f"Epoch [{epoch:>{len(str(num_epochs))}}/{num_epochs}]  "
            f"Loss: {train_loss:.4f}  "
            f"Train Acc: {train_acc:.4f}  "
            f"Val Acc: {val_acc:.4f}"
            f"{saved}"
        )

    return history, best_val_acc


In [ ]:
def evaluate(model, loader, device):
    """
    Compute classification accuracy over a DataLoader.

    Parameters
    ----------
    model  : nn.Module
    loader : DataLoader
    device : torch.device

    Returns
    -------
    accuracy : float in [0, 1]
    """
    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images).squeeze(1)        # (batch,)
            preds   = (outputs >= 0.5).float()        # threshold at 0.5
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    return correct / total

print("evaluate() helper defined.")


evaluate() helper defined.


In [ ]:
history, best_val_acc = train(
    model       = model,
    train_loader= train_loader,
    val_loader  = val_loader,
    criterion   = criterion,
    optimizer   = optimizer,
    num_epochs  = NUM_EPOCHS,
    device      = DEVICE,
    checkpoint_path = CHECKPOINT_PATH,
)

print(f"\nTraining complete.")
print(f"Best validation accuracy : {best_val_acc:.4f}")
print(f"Checkpoint saved to      : {CHECKPOINT_PATH}")


Batch 0/4096 | Loss: 0.7368 | Batch Acc: 0.4375
Batch 100/4096 | Loss: 0.4492 | Batch Acc: 0.8125
Batch 200/4096 | Loss: 0.4717 | Batch Acc: 0.7969
Batch 300/4096 | Loss: 0.4918 | Batch Acc: 0.7969
Batch 400/4096 | Loss: 0.4659 | Batch Acc: 0.7812
Batch 500/4096 | Loss: 0.4691 | Batch Acc: 0.7812
Batch 600/4096 | Loss: 0.5067 | Batch Acc: 0.7812
Batch 700/4096 | Loss: 0.6177 | Batch Acc: 0.6875
Batch 800/4096 | Loss: 0.4552 | Batch Acc: 0.8125
Batch 900/4096 | Loss: 0.4632 | Batch Acc: 0.8125
Batch 1000/4096 | Loss: 0.5753 | Batch Acc: 0.7344
Batch 1100/4096 | Loss: 0.5676 | Batch Acc: 0.6719
Batch 1200/4096 | Loss: 0.4971 | Batch Acc: 0.7812
Batch 1300/4096 | Loss: 0.5106 | Batch Acc: 0.7969
Batch 1400/4096 | Loss: 0.6024 | Batch Acc: 0.7188
Batch 1500/4096 | Loss: 0.4524 | Batch Acc: 0.7969
Batch 1600/4096 | Loss: 0.5113 | Batch Acc: 0.7031
Batch 1700/4096 | Loss: 0.6375 | Batch Acc: 0.7656
Batch 1800/4096 | Loss: 0.6031 | Batch Acc: 0.6875
Batch 1900/4096 | Loss: 0.5203 | Batch Acc:

# Test

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────────
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

all_probs  = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        probs  = model(images).squeeze(1).cpu().numpy()   # raw probabilities
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.numpy().tolist())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# ── Metrics ────────────────────────────────────────────────────────────────────
test_acc = float(((all_probs >= 0.5).astype(float) == all_labels).mean())
test_auc = float(roc_auc_score(all_labels, all_probs))

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test AUC-ROC  : {test_auc:.4f}")


Test Accuracy : 0.7631
Test AUC-ROC  : 0.8506


# Results

In [ ]:
print("=" * 47)
print(f"{'Backbone':<22} {'Test Acc':>10} {'Test AUC':>10}")
print("-" * 47)
print(f"{'vgg16':<22} {test_acc:>10.4f} {test_auc:>10.4f}")
print("=" * 47)


Backbone                 Test Acc   Test AUC
-----------------------------------------------
vgg16                      0.7631     0.8506
